# ResNet18 — first full 26-leaf baseline

Bu notebook ilk raporlanabilir **baseline** eğitimini hazırlar. Model 26 leaf etiketi tahmin eder; aynı tahminlerden sekiz parent-group metriği türetilir. Sabit `practical_leakage_reduced_split_v1.csv` dışında split kullanılamaz.

Bu bir klinik tanı sistemi değildir. Test seti model veya hiperparametre seçmek için kullanılmaz; checkpoint yalnız validation macro-F1 ile seçilir.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time

assert 'google.colab' in sys.modules, 'Bu notebook Google Colab için hazırlanmıştır.'
from google.colab import drive
drive.mount('/content/drive')
print('Drive bağlandı.')


## 1. Proje, ZIP ve kalıcı çıktı yolları

130.000 küçük dosyayı Drive üzerinden tek tek okumak yavaş olduğu için `Multi Cancer.zip` Colab'ın geçici `/content` diskine açılır. Runtime kapanırsa yerel kopya silinir; Drive'daki ZIP ve eğitim çıktıları korunur.


In [ ]:
REPOSITORY_URL = 'https://github.com/YusufOck/Neural-Network-and-Genens-s-Algor-thm-Project-Erasmus-.git'
PROJECT_ROOT = Path('/content/NNGA')
ZIP_PATH = Path('/content/drive/MyDrive/NNGA_DATA/Multi Cancer.zip')
LOCAL_DATA_BASE = Path('/content/NNGA_DATA')
EXTRACT_MARKER = LOCAL_DATA_BASE / '.multi_cancer_extract_complete'
OUTPUT_DIR = Path('/content/drive/MyDrive/NNGA_OUTPUTS/resnet18_26leaf_baseline_v1')
LOG_PATH = Path('/content/drive/MyDrive/NNGA_OUTPUTS/resnet18_26leaf_baseline_v1_console.log')
FROZEN_SPLIT_SHA256 = '999bf7a9bd1bba9f2957a9db0e13ae72adb3ecb8fdbc1f33cce40b083c1cf171'
FROZEN_TRAINER_SHA256 = '1839b0c65a26f0ea7d8719bcd58238b8c73eb3e78632b26e377937e018474490'
MINIMUM_TRAINER_COMMIT = '3e800b9'
EXPECTED_GROUPS = (
    'ALL', 'Brain Cancer', 'Breast Cancer', 'Cervical Cancer',
    'Kidney Cancer', 'Lung and Colon Cancer', 'Lymphoma', 'Oral Cancer',
)

if (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
commit = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
ancestor = subprocess.run(
    ['git', '-C', str(PROJECT_ROOT), 'merge-base', '--is-ancestor', MINIMUM_TRAINER_COMMIT, 'HEAD']
).returncode == 0
assert ancestor, f'Trainer hardening commit bulunamadı; HEAD={commit}'
trainer_hash = hashlib.sha256((PROJECT_ROOT / 'src/train.py').read_bytes()).hexdigest()
assert trainer_hash == FROZEN_TRAINER_SHA256, f'Trainer SHA-256 uyuşmuyor: {trainer_hash}'

assert ZIP_PATH.is_file(), f'Dataset ZIP bulunamadı: {ZIP_PATH}'
free_gib = shutil.disk_usage('/content').free / (1024 ** 3)
print(f'ZIP: {ZIP_PATH} ({ZIP_PATH.stat().st_size / (1024 ** 3):.2f} GiB)')
print(f'Colab yerel boş alanı: {free_gib:.2f} GiB')
assert free_gib >= 12.0, 'Dataset açma ve eğitim için en az 12 GiB yerel boş alan gerekli.'

if not EXTRACT_MARKER.is_file():
    LOCAL_DATA_BASE.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    print('ZIP Colab diskine çıkarılıyor; birkaç dakika çıktı görünmeyebilir...')
    subprocess.run(['unzip', '-q', '-o', str(ZIP_PATH), '-d', str(LOCAL_DATA_BASE)], check=True)
    EXTRACT_MARKER.write_text('complete\n', encoding='utf-8')
    print(f'ZIP çıkarma tamamlandı: {(time.monotonic() - started) / 60:.1f} dakika')
else:
    print('Tamamlanmış yerel çıkarma bulundu; ZIP yeniden açılmadı.')

dataset_candidates = (
    LOCAL_DATA_BASE / 'Multi Cancer' / 'Multi Cancer',
    LOCAL_DATA_BASE / 'Multi Cancer',
    LOCAL_DATA_BASE,
)
DATASET_ROOT = next(
    (path for path in dataset_candidates if all((path / group).is_dir() for group in EXPECTED_GROUPS)),
    None,
)
assert DATASET_ROOT is not None, 'Sekiz ana klasörü içeren dataset kökü bulunamadı.'
SPLIT_MANIFEST = PROJECT_ROOT / 'research/splits/practical_leakage_reduced_split_v1.csv'
assert SPLIT_MANIFEST.is_file(), f'Split manifest bulunamadı: {SPLIT_MANIFEST}'
split_hash = hashlib.sha256(SPLIT_MANIFEST.read_bytes()).hexdigest()
assert split_hash == FROZEN_SPLIT_SHA256, f'Split SHA-256 uyuşmuyor: {split_hash}'
Path('/content/drive/MyDrive/NNGA_OUTPUTS').mkdir(parents=True, exist_ok=True)
print('Git commit:', commit)
print('Trainer SHA-256:', trainer_hash)
print('Dataset:', DATASET_ROOT)
print('Split SHA-256:', split_hash)
print('Kalıcı output:', OUTPUT_DIR)


## 2. GPU ve eğitim ortamı kontrolü

Colab menüsünden **Runtime → Change runtime type → T4 GPU** seç. Bu hücre CUDA'yı, gerekli Python paketlerini ve GPU adını doğrular; model eğitmez.


In [ ]:
import matplotlib, numpy, PIL, sklearn, torch, torchvision
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('NumPy:', numpy.__version__, '| Pillow:', PIL.__version__)
print('scikit-learn:', sklearn.__version__, '| matplotlib:', matplotlib.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK')
assert torch.cuda.is_available(), 'GPU seçilmemiş; T4 runtime aç.'


## 3. Dondurulmuş baseline sözleşmesi

- Model: ImageNet-pretrained ResNet18
- Hedef: 26 leaf; sekiz parent sonuç aynı tahminlerden türetilir
- Split: frozen practical manifest, 91.000 / 19.500 / 19.500
- Girdi: 224×224, ImageNet normalization, minimal train-only augmentation
- Eğitim: yalnız classifier head, AdamW, learning rate 0.001, weight decay 0.0001
- Bütçe: en fazla 15 epoch; validation macro-F1 dört epoch iyileşmezse erken durur
- Test: yalnız validation ile seçilen en iyi checkpoint yüklendikten sonra değerlendirilir

Tahmini süre T4 ve Colab yoğunluğuna göre yaklaşık 1–3 saattir; erken durdurma daha kısa bitirebilir. Her tamamlanan epoch Drive'a checkpoint edilir. Runtime kesilirse aynı hücre, mevcut sözleşme değişmemişse son tamamlanan epoch'tan devam eder.

Bu hücre gerçek tam eğitimi başlatır. Yalnız Codex ile sözleşmeyi onayladıktan sonra `AUTHORIZE_FULL_BASELINE = True` yap.


In [ ]:
AUTHORIZE_FULL_BASELINE = False
assert AUTHORIZE_FULL_BASELINE, 'Tam baseline kilitli; önce sözleşmeyi Codex ile onayla.'
assert not (OUTPUT_DIR / 'test_metrics.json').exists(), 'Bu baseline zaten tamamlanmış; yeniden çalıştırma.'
RESUME_RUN = (OUTPUT_DIR / 'last_checkpoint.pth').is_file()
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()) and not RESUME_RUN:
    raise RuntimeError('Output klasörü boş değil ve geçerli resume checkpoint yok; Codex ile incele.')

command = [
    sys.executable, '-u', str(PROJECT_ROOT / 'src/train.py'),
    '--data-dir', str(DATASET_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--model', 'resnet18',
    '--target-level', 'subclass',
    '--split-manifest', str(SPLIT_MANIFEST),
    '--pretrained',
    '--epochs', '15',
    '--early-stopping-patience', '4',
    '--batch-size', '32',
    '--image-size', '224',
    '--normalization', 'imagenet',
    '--augmentation', 'minimal',
    '--learning-rate', '0.001',
    '--weight-decay', '0.0001',
    '--num-workers', '4',
    '--seed', '42',
    '--experiment-type', 'baseline',
]
if RESUME_RUN:
    command.append('--resume')
    print('Resume checkpoint bulundu; tamamlanan epochlar yeniden çalışmayacak.')
print('Baseline komutu:', ' '.join(command))
print('Konsol logu:', LOG_PATH)
with LOG_PATH.open('w', encoding='utf-8') as log_handle:
    process = subprocess.Popen(
        command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='')
        log_handle.write(line)
        log_handle.flush()
    return_code = process.wait()
print('RETURN CODE:', return_code)
if return_code != 0:
    raise RuntimeError('Baseline tamamlanamadı. Output klasörünü değiştirme; logu Codex’e gönder.')


## 4. Tamamlanma ve artifact kontrolü

Başarılı koşuda aşağıdaki hücre 19.500 test tahminini, frozen split hash'ini ve bütün temel çıktıları doğrular. Bize önce `test_metrics.json`, `history.csv` ve konsol logunu gönder. Model ağırlığını indirmen gerekmez; Drive'da koru.


In [ ]:
required_outputs = [
    'best_model.pth', 'last_checkpoint.pth', 'history.csv', 'test_metrics.json', 'test_predictions.csv',
    'classification_reports.json', 'leaf_confusion_matrix.png',
    'parent_confusion_matrix.png', 'learning_curves.png', 'split_manifest.csv',
]
missing = [name for name in required_outputs if not (OUTPUT_DIR / name).is_file()]
assert not missing, f'Eksik baseline çıktıları: {missing}'
copied_split_hash = hashlib.sha256((OUTPUT_DIR / 'split_manifest.csv').read_bytes()).hexdigest()
assert copied_split_hash == FROZEN_SPLIT_SHA256, "Run içindeki split manifest hash'i yanlış."
metrics = json.loads((OUTPUT_DIR / 'test_metrics.json').read_text(encoding='utf-8'))
prediction_rows = sum(1 for _ in (OUTPUT_DIR / 'test_predictions.csv').open(encoding='utf-8')) - 1
assert metrics['target_level'] == 'subclass'
assert metrics['split_sizes'] == {'train': 91000, 'val': 19500, 'test': 19500}
assert prediction_rows == 19500, f'Beklenen 19.500 test tahmini, bulunan {prediction_rows}'
print('Durum: resnet18_26leaf_baseline_v1_complete')
print('Best epoch:', metrics['best_epoch'])
print('Leaf metrics:', metrics['leaf_metrics'])
print('Parent metrics:', metrics['parent_metrics'])
print('Prediction rows:', prediction_rows)
print('Split SHA-256:', copied_split_hash)
print('Codex’e ver:', OUTPUT_DIR / 'test_metrics.json')
print('Codex’e ver:', OUTPUT_DIR / 'history.csv')
print('Codex’e ver:', LOG_PATH)
